In [16]:
from openai import OpenAI
import gradio as gr
import os
import json
import requests
import sqlite3

from dotenv import load_dotenv
from services import (
    insert_medicines_from_csv,
    get_medicine_info,
    drug_lookup,
)

from services.phase2_medicine_llm_schema import MEDICINE_TOOLS

In [2]:
# One-time database setup
insert_medicines_from_csv()

✓ Database schema created successfully!
✓ Data imported successfully!
✓ Total medicines inserted: 11825


Test Query

In [3]:
conn = sqlite3.connect('db/medicine_info.db')
cursor = conn.cursor()

# Fetch and print one record to verify data insertion
cursor.execute('''
    SELECT *
    FROM medicines
    LIMIT 1
        ''')
        
result = cursor.fetchone()

print(result)
        
# Fetch and print the total count of records in the medicines table        
cursor.execute('''
    SELECT count(*)
    FROM medicines
        ''')
        
result = cursor.fetchone()
print(result)

conn.close()

(1, 'Avastin 400mg Injection', 'Bevacizumab (400mg)', 'Roche Products India Pvt Ltd', ' Cancer of colon and rectum Non-small cell lung cancer Kidney cancer Brain tumor Ovarian cancer Cervical cancer', 'Rectal bleeding Taste change Headache Nosebleeds Back pain Dry skin High blood pressure Protein in urine Inflammation of the nose')
(11825,)


In [4]:
results = get_medicine_info("Aspirin")

for result in results:
    print(result)

{'medicine_name': 'Aztor Asp 75 Capsule', 'composition': 'Atorvastatin (10mg) + Aspirin (75mg)', 'manufacturer': 'Sun Pharmaceutical Industries Ltd', 'uses': 'Treatment of Prevention of heart attack and stroke', 'side_effects': 'Abdominal pain Constipation Flatulence Increased liver enzymes Hepatitis viral infection of liver Reye s syndrome like symptoms'}
{'medicine_name': 'Aztogold 20 Capsule', 'composition': 'Aspirin (75mg) + Atorvastatin (20mg) + Clopidogrel (75mg)', 'manufacturer': 'Sun Pharmaceutical Industries Ltd', 'uses': 'Prevention of Heart attack', 'side_effects': 'Increased bleeding tendency Abdominal pain Indigestion Bruise Nosebleeds Gastrointestinal bleeding Diarrhea'}
{'medicine_name': 'Atchol-ASP Capsule', 'composition': 'Atorvastatin (10mg) + Aspirin (75mg)', 'manufacturer': 'Aristo Pharmaceuticals Pvt Ltd', 'uses': 'Treatment of Prevention of heart attack and stroke', 'side_effects': 'Abdominal pain Constipation Flatulence Increased liver enzymes Hepatitis viral inf

In [5]:
result = drug_lookup("Aspirin")
print(result)

Inside the drug_lookup function
Querying OpenFDA API for generic name: Aspirin
Successfully retrieved data from OpenFDA API
{'brand_name': 'Low Dose Aspirin', 'generic_name': 'ASPIRIN', 'active_ingredients': ['Active ingredient (in each tablet) Aspirin 81 mg (NSAID)* *nonsteroidal anti-inflammatory drug'], 'purpose': 'Purpose Pain reliever', 'indications': 'Uses for the temporary relief of minor aches and pains or as recommended by your doctor. Because of its delayed action, this product will not provide fast relief of headaches or other symptoms needing immediate relief. ask your doctor about other uses for safety coated 81 mg aspirin', 'warnings': "Warnings Reye's syndrome : Children and teenagers who have or are recovering from chicken pox or flu-like symptoms should not use this product. When using this product, if changes in behavior with nausea and vomiting occur, consult a doctor because these symptoms could be an early sign of Reye's syndrome, a rare but serious illness. Allerg

In [6]:
mistral_model = "mistral:latest"
llama_model = "llama3.2:1b"
tiny_model = "tinyllama:latest"
neural_model = "neural-chat:latest"

# Defining openAI client for local ollama models

# client = OpenAI(
#     base_url="http://localhost:11434/v1",
#     api_key="ollama"  # required but can be any string
# )

# Defining openAI client for OpenAI hosted models
load_dotenv()  # Load environment variables from .env file

gpt_model = "gpt-4o-mini"
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# curr_model = llama_model
curr_model = gpt_model

In [7]:
system_prompt = """
You are a helpful pharmaceutical information assistant. Your role is to help users find accurate and reliable information about medications, including their brand names, generic names, active ingredients, purposes, indications, and important safety warnings.

You have access to two tools:
1. get_medicine_info: Search a local database of 11,000+ medicines by brand name or generic name. Returns medicine details including manufacturer, uses, and side effects.
2. drug_lookup: Query the OpenFDA API for official FDA drug label information including warnings, indications, active ingredients, and safety data.

When a user asks about a medication:
1. First use get_medicine_info to find the medicine and its basic details
2. Then use drug_lookup with the generic name to retrieve FDA safety warnings and official label information
3. Combine information from both sources to provide a comprehensive response
4. Present the information in a clear, organized manner
5. Always emphasize important warnings and safety information prominently
6. If a medication is not found, suggest alternative search terms or recommend consulting a healthcare professional
7. Never provide medical advice—only factual information from the databases
8. If information is missing or marked as "N/A", acknowledge this limitation
9. Remind users to always consult with healthcare professionals before taking any medication

Always prioritize accuracy and user safety in your responses.
"""

In [8]:
tools = MEDICINE_TOOLS
tools

[{'type': 'function',
  'function': {'name': 'get_medicine_info',
   'description': 'Find medicine information by brand name or generic name...',
   'parameters': {'type': 'object',
    'properties': {'search_term': {'type': 'string',
      'description': 'The medicine brand name or generic name...'}},
    'required': ['search_term']}}},
 {'type': 'function',
  'function': {'name': 'drug_lookup',
   'description': 'Query the OpenFDA API for drug label information...',
   'parameters': {'type': 'object',
    'properties': {'generic_name': {'type': 'string',
      'description': 'The generic name of the drug to look up...'}},
    'required': ['generic_name']}}}]

In [12]:
def handle_tool_calls(message):
    
    print("Inside handle_tool_calls function")
    print("Received message:", message)

    responses = []

    for tool_call in message.tool_calls:
        print("Processing tool call:", tool_call)

        tool_result = None

        if tool_call.function.name == "drug_lookup":
            try:
                print("Parsing tool call arguments:", tool_call.function.arguments)
                args = json.loads(tool_call.function.arguments)
            except json.JSONDecodeError as e:
                print("Error parsing tool call arguments:", e)
                args = {}

            generic_name = args.get("generic_name")

            if not generic_name:
                tool_result = {"error": "No generic name provided."}
            else:   
                print(f"Looking up drug information for: {generic_name}")
                tool_result = drug_lookup(generic_name)
        
        elif tool_call.function.name == "get_medicine_info":
            try:
                print("Parsing tool call arguments:", tool_call.function.arguments)
                args = json.loads(tool_call.function.arguments)
            except json.JSONDecodeError as e:
                print("Error parsing tool call arguments:", e)
                args = {}

            search_term = args.get("search_term")

            if not search_term:
                tool_result = {"error": "No search term provided."}
            else:   
                print(f"Getting medicine info for: {search_term}")
                tool_result = get_medicine_info(search_term)

        else:
            print(f"Unknown tool called: {tool_call.function.name}")
            tool_result = {"error": f"Unknown tool called: {tool_call.function.name}"}

        response = {
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(tool_result)
            }

        responses.append(response)
    return responses
            

In [13]:
def med_tool_chat(message, history):

    print("Inside the med_tool_chat function")
    messages = [{"role": "system", "content": system_prompt}]
    
    for h in history:
        content = h["content"]
        if isinstance(content, list):
            content = content[0]["text"] if content else ""
        messages.append({
            "role": h["role"], 
            "content": content
            })
    
    messages.append({"role": "user", "content": message})

    response = client.chat.completions.create(
        model=curr_model,
        messages=messages,
        temperature=0,
        tools=tools
    )

    max_iterations = 5
    iteration = 0

    while response.choices[0].message.tool_calls and iteration < max_iterations:
        iteration += 1

        assistant_message = response.choices[0].message

        messages.append({
            "role": "assistant",
            "content": assistant_message.content or "",
            "tool_calls": assistant_message.tool_calls

        })

        tool_responses = handle_tool_calls(assistant_message)
        messages.extend(tool_responses)

        # for msg in messages:
        #     print(f"{msg['role']}: {msg['content']}")

        response = client.chat.completions.create(
            model=curr_model,
            messages=messages,
            temperature=0,
            tools=tools
        )
    
    if iteration == max_iterations and response.choices[0].message.tool_calls:
        print("Max iterations reached while processing tool calls. Some tool calls may not have been handled.") 

    return response.choices[0].message.content or ""

In [15]:
gr.ChatInterface(fn=med_tool_chat).launch()

* Running on local URL:  http://127.0.0.1:7870
* To create a public link, set `share=True` in `launch()`.


Inside the med_tool_chat function
Inside the med_tool_chat function
Inside handle_tool_calls function
Received message: ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_BUP01waTG2bmkFVG4Ox3kahE', function=Function(arguments='{"search_term": "Aspirin"}', name='get_medicine_info'), type='function'), ChatCompletionMessageFunctionToolCall(id='call_QWEqni7U8bFsd9UD145g75Ju', function=Function(arguments='{"search_term": "Ibuprofen"}', name='get_medicine_info'), type='function'), ChatCompletionMessageFunctionToolCall(id='call_ktzERXxvSEusnNFk5yU8gSJO', function=Function(arguments='{"search_term": "Naproxen"}', name='get_medicine_info'), type='function')])
Processing tool call: ChatCompletionMessageFunctionToolCall(id='call_BUP01waTG2bmkFVG4Ox3kahE', function=Function(arguments='{"search_term": "Aspirin"}', name='get_medicine_info'), type='function')
Parsing tool call a